In [1]:
import torch, os, re
from torch import nn
from PIL import Image
from typing import Dict
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from transformers import ViTImageProcessor, AutoTokenizer
from project.multimodal_training_utils import MultimodalConfig, MultimodalModel,  train_model, custom_collate_fn

c:\Users\wolfg\OneDrive\Documents\CivitAI\Project Work\research\multimodalCheck\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Notebook Purpose:
<b> This is notebook number 3d </b>

Gives code for training multimodal Model

### Notebook Order
1. getData
2. downloadData
3. trainResNetModel | trainPromptTransformerClassifier | trainViTClassifier | trainYoloCLS | trainMultimodalModel
4. localMixtureEval |  localModelEval

define dataset in notebook to avoid idx issues

In [2]:
class MultimodalDataset(Dataset):
    def __init__(self, root_dir: str, models: Dict[str, Dict[str, str]], label2id: Dict[str, int] = None):
        self.root_dir = root_dir
        self.models = models
        self.label2id = label2id if label2id is not None else {'PG': 0, 'PG13': 1, 'R': 2, 'X': 3, 'XXX': 4}
        self.data_ = self._load_data()
        self.image_paths = self.data_['image_path']
        self.texts = self.data_['text']
        self.labels = self.data_['labels']
        self.processors = self.get_image_text_processors()

    def _load_data(self):
        data_ = {'image_path': [], 'text': [], 'labels': []}
        for label_name in os.listdir(self.root_dir):
            label_dir = os.path.join(self.root_dir, label_name)
            if os.path.isdir(label_dir):
                for file_name in os.listdir(label_dir):
                    if file_name.endswith('.jpg'):
                        img_path = os.path.join(label_dir, file_name)
                        txt_path = os.path.join(label_dir, file_name.replace('.jpg', '.txt'))
                        if os.path.exists(txt_path):
                            with open(txt_path, 'r') as file:
                                text = file.read()
                            data_['image_path'].append(img_path)
                            data_['text'].append(text)
                            data_['labels'].append(self.label2id.get(label_name, -1))
        return data_

    def get_image_text_processors(self):
        processors = {}
        for model_type, model_dict in self.models.items():
            if model_type == 'resnet':
                for model, dir in model_dict.items():
                    processors[model] = transforms.Compose([
                        transforms.Resize((224, 224)),
                        transforms.ToTensor(),
                        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                    ])
            elif model_type == 'vit':
                for model, dir in model_dict.items():
                    processors[model] = ViTImageProcessor.from_pretrained(dir)
            elif model_type == 'nlp':
                for model, dir in model_dict.items():
                    processors[model] = AutoTokenizer.from_pretrained(dir)
        return processors

    @staticmethod
    def clean_text(text: str) -> str:
        text = str(text)
        cleaned_text = re.sub(r"[():<>[\]]", " ", text)
        cleaned_text = cleaned_text.replace("\n", " ")
        cleaned_text = re.sub(r"\s+", " ", cleaned_text)
        cleaned_text = re.sub(r"\s*,\s*", ", ", cleaned_text)
        return cleaned_text.strip()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        if isinstance(idx, list):  # Added to handle list indices
            raise TypeError("Dataset indices should be integers, not lists")

        image = Image.open(self.image_paths[idx]).convert('RGB')
        cleaned_text = self.clean_text(self.texts[idx])

        sample = {}
        for model_type, model_dict in self.models.items():
            for model_name in model_dict:
                if model_type == 'resnet':
                    sample[model_name] = self.processors[model_name](image)
                elif model_type == 'vit':
                    sample[model_name] = self.processors[model_name](image, return_tensors="pt")['pixel_values']
                elif model_type == 'nlp':
                    sample[model_name] = dict(self.processors[model_name](cleaned_text, return_tensors="pt", padding=True, truncation=True))

        sample['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return sample


### Instantiate model
note: current model ONLY supports resnets, vit, and bert/roberta

In [3]:
resnet_paths = {'resnet18': './models/baseresNet18/',
                'resnet50': './models/baseresNet50/'}
vit_paths = {'vit': './models/vitRater'}
bert_paths = {'bert': './models/promptBert',
              "roberta": './models/promptRoberta/'}

config = MultimodalConfig(
    resnet_model_paths=resnet_paths,
    vit_model_paths=vit_paths,
    nlp_transformers_model_paths=bert_paths
)

In [5]:
model = MultimodalModel(config)

## Get dataloader

In [9]:
# Assuming MultimodalDataset is defined
train_dataset = MultimodalDataset('./data/datasets/may/train', models=config.models)
val_dataset = MultimodalDataset('./data/datasets/may/val', models=config.models)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)

In [10]:
# Define the optimizer
optimizer = optim.Adam(model.mlp.parameters(), lr=1e-4)

# Define the loss function (already included in your model)
loss_fn = nn.CrossEntropyLoss()

In [11]:
model = train_model(model, train_dataloader, val_dataloader, optimizer, num_epochs=25, save_dir='./models/multimodal/')

Epoch 1/25 - Training:   0%|          | 0/668 [00:00<?, ?it/s]c:\Users\wolfg\OneDrive\Documents\CivitAI\Project Work\research\multimodalCheck\venv\Lib\site-packages\transformers\models\vit\modeling_vit.py:253: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  context_layer = torch.nn.functional.scaled_dot_product_attention(


Epoch 1/25, Training Loss: 0.2987222051370644


Epoch 1/25, Validation Loss: 0.20830789740568312, Validation Accuracy: 0.9334394059931053
New best model saved with validation loss: 0.20830789740568312


Epoch 2/25, Training Loss: 0.22650200889228347


Epoch 2/25, Validation Loss: 0.18807106977525972, Validation Accuracy: 0.9350304958896845
New best model saved with validation loss: 0.18807106977525972


...

Epoch 7/25, Validation Loss: 0.18212895302330853, Validation Accuracy: 0.9363564041368337


Epoch 8/25, Training Loss: 0.16052118188863296


Epoch 8/25, Validation Loss: 0.17277321552584476, Validation Accuracy: 0.9427207637231504


Epoch 9/25, Training Loss: 0.1508852826141401


Epoch 9/25, Validation Loss: 0.17478930476142324, Validation Accuracy: 0.94351630867144
Early stopping triggered.


## Save model

In [13]:
# Save the model - note: we had an issue with saving this way, I believe it was fixed, but be wary. It may be preudent to save
#torch.save(model.mlp.state_dict())
model.save_pretrained('./models/multimodal_model')

MultimodalModel(
  (resnet_models): ModuleDict(
    (resnet18): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        